# Chia dữ liệu thành ba tập theo thời gian

Đầu vào: `data/interim/hotel_bookings_clean.csv` (kết quả của notebook làm sạch).
Đầu ra: `data/processed/train.csv`, `val.csv`, `test.csv`.

| Tập | Vai trò | Khoảng ngày đến |
|---|---|---|
| Huấn luyện | Máy học từ đây | 2015-07 đến 2016-08 |
| Kiểm định | Thử, so sánh mô hình, chỉnh tham số, chọn ngưỡng | 2016-09 đến 2016-12 |
| Kiểm tra | Chấm điểm cuối cùng, **chỉ chạy một lần** | 2017-01 đến 2017-08 |

Chia theo thời gian (không chia ngẫu nhiên) vì ngoài thực tế khách sạn dùng dữ liệu quá khứ để đoán tương lai. Chia ngẫu nhiên sẽ cho điểm đẹp giả (đây cũng là thực nghiệm E5).

Notebook này **không biến đổi dữ liệu**: các file ra giữ nguyên 25 cột của file sạch. Mọi bước học số liệu (chuẩn hóa, mã hóa...) làm sau, trong Pipeline, chỉ học trên tập huấn luyện.

In [1]:
import pandas as pd

SRC = "../data/interim/hotel_bookings_clean.csv"
OUT_DIR = "../data/processed"

# Ranh giới các tập, tính theo năm-tháng của NGÀY ĐẾN (arrival date), gồm cả hai đầu
SPLITS = {
    "train": (201507, 201608),
    "val":   (201609, 201612),
    "test":  (201701, 201708),
}

## Đọc dữ liệu và tạo ngày đến

In [2]:
df = pd.read_csv(SRC)

MONTHS = {m: i for i, m in enumerate(
    ["January", "February", "March", "April", "May", "June", "July", "August",
     "September", "October", "November", "December"], 1)}

# Ngày đến dùng để chia; chỉ nằm trong biến tạm, KHÔNG thêm vào file đầu ra
arrival = pd.to_datetime(dict(
    year=df["arrival_date_year"],
    month=df["arrival_date_month"].map(MONTHS),
    day=df["arrival_date_day_of_month"]))
ym = arrival.dt.year * 100 + arrival.dt.month

print(f"{len(df):,} dòng, ngày đến từ {arrival.min().date()} đến {arrival.max().date()}")

86,617 dòng, ngày đến từ 2015-07-01 đến 2017-08-31


## Gán từng dòng vào một tập

In [3]:
split = pd.Series(pd.NA, index=df.index, dtype="object")
for name, (lo, hi) in SPLITS.items():
    split[ym.between(lo, hi)] = name

# Kiểm tra: mỗi dòng thuộc đúng một tập, không dòng nào bị bỏ sót
assert split.notna().all(), f"{split.isna().sum()} dòng không thuộc tập nào"
parts = {name: df[split == name] for name in SPLITS}
assert sum(len(p) for p in parts.values()) == len(df)

# Kiểm tra thứ tự thời gian: tập sau phải đến sau tập trước hoàn toàn
rng = {name: (arrival[split == name].min(), arrival[split == name].max()) for name in SPLITS}
assert rng["train"][1] < rng["val"][0] and rng["val"][1] < rng["test"][0]
print("OK: ba tập không giao nhau, đủ", len(df), "dòng, đúng thứ tự thời gian")

OK: ba tập không giao nhau, đủ 86617 dòng, đúng thứ tự thời gian


## Tóm tắt ba tập

In [4]:
summary = pd.DataFrame({
    "dòng": {n: len(p) for n, p in parts.items()},
    "tỷ lệ": {n: len(p) / len(df) for n, p in parts.items()},
    "từ ngày": {n: rng[n][0].date() for n in parts},
    "đến ngày": {n: rng[n][1].date() for n in parts},
    "tỷ lệ hủy": {n: p["is_canceled"].mean() for n, p in parts.items()},
})
summary["tỷ lệ"] = summary["tỷ lệ"].map("{:.1%}".format)
summary["tỷ lệ hủy"] = summary["tỷ lệ hủy"].map("{:.1%}".format)
summary

,dòng,tỷ lệ,từ ngày,đến ngày,tỷ lệ hủy
train,40688,47.0%,2015-07-01,2016-08-31,24.0%
val,14400,16.6%,2016-09-01,2016-12-31,28.4%
test,31529,36.4%,2017-01-01,2017-08-31,32.0%


Tỷ lệ hủy theo khách sạn trong từng tập (cần cho thực nghiệm E6, huấn luyện ở khách sạn này và kiểm tra ở khách sạn kia):

In [5]:
(df.assign(tap=split)
   .groupby(["tap", "hotel"])["is_canceled"]
   .agg(so_dong="size", ty_le_huy="mean")
   .round(3))

so_dong  ty_le_huy
tap   hotel                           
test  City Hotel      20014      0.341
      Resort Hotel    11515      0.284
train City Hotel      23760      0.262
      Resort Hotel    16928      0.209
val   City Hotel       9262      0.319
      Resort Hotel     5138      0.221

## Ghi ra file

In [6]:
for name, part in parts.items():
    part.to_csv(f"{OUT_DIR}/{name}.csv", index=False)
    print(f"Đã ghi {OUT_DIR}/{name}.csv: {len(part):,} dòng x {part.shape[1]} cột")

Đã ghi ../data/processed/train.csv: 40,688 dòng x 25 cột
Đã ghi ../data/processed/val.csv: 14,400 dòng x 25 cột
Đã ghi ../data/processed/test.csv: 31,529 dòng x 25 cột


## Quy tắc từ đây trở đi

- Mọi EDA, chọn đặc trưng, tinh chỉnh tham số, chọn ngưỡng: chỉ dùng `train.csv` và `val.csv`.
- `test.csv` chưa mở cho đến bước đánh giá cuối cùng, và chỉ chạy một lần.
- Tỷ lệ hủy tăng dần qua các tập (khoảng 24% rồi 28% rồi 32%): dữ liệu có trôi theo thời gian, đây là điểm cần nhắc trong phần đánh giá.